<a href="https://colab.research.google.com/github/dawamukherjee/delhi-aqi-predictor/blob/main/02_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Delhi AQI Predictor: model and purifier rule
**Track:** AI + Climate | **Team:** Aahan Chaurasia, Tanveer Kaliraman, Dawa Mukherjee

*This project is a prototype and is not medical advice.*

We forecast New Delhi PM2.5 six hours ahead, then use the forecast to decide when to run an air purifier. The data comes from Open-Meteo (model estimates, not sensor readings) and is loaded from our GitHub repo. We train on older data and test on the last 60 days, which the model never saw.

## 1. Baseline and linear regression
The baseline guesses that PM2.5 six hours from now will equal PM2.5 right now. Average error is in µg/m³, so lower is better.

In [1]:
import pandas as pd, numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

# Load the data you uploaded
df = pd.read_csv("https://raw.githubusercontent.com/dawamukherjee/delhi-aqi-predictor/main/delhi_pm25_weather_hourly.csv", parse_dates=["time"])

# Make the "answer" column: PM2.5 six hours from now
H = 6
df["target"] = df["pm2_5"].shift(-H)
df["pm_1h_ago"] = df["pm2_5"].shift(1)
df["pm_3h_ago"] = df["pm2_5"].shift(3)
df["hour"] = df["time"].dt.hour
d = df.dropna().reset_index(drop=True)

# Train on older data, test on the most recent 60 days
cut = d["time"].max() - pd.Timedelta(days=60)
train = d[d["time"] <= cut]
test = d[d["time"] > cut]

features = ["pm2_5", "pm_1h_ago", "pm_3h_ago", "pm10", "temperature_2m",
            "relative_humidity_2m", "wind_speed_10m", "precipitation", "hour"]

# Baseline: "6 hours from now will be the same as right now"
baseline = mean_absolute_error(test["target"], test["pm2_5"])

# Train the model, then predict on the test data
model = LinearRegression().fit(train[features], train["target"])
pred = model.predict(test[features])
mine = mean_absolute_error(test["target"], pred)

print(f"Baseline error: {baseline:.1f} µg/m³")
print(f"Model error:    {mine:.1f} µg/m³")

Baseline error: 22.9 µg/m³
Model error:    18.1 µg/m³


In [2]:
import requests, pandas as pd

AQ_URL = "https://air-quality-api.open-meteo.com/v1/air-quality?latitude=28.6214&longitude=77.2148&hourly=pm10,pm2_5&timezone=auto&past_days=365&forecast_days=1"
aq = pd.DataFrame(requests.get(AQ_URL).json()["hourly"])
aq["time"] = pd.to_datetime(aq["time"])

W_URL = "https://archive-api.open-meteo.com/v1/archive?latitude=28.6214&longitude=77.2148&start_date=2025-11-14&end_date=2026-11-09&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m,precipitation&timezone=auto"
wx = pd.DataFrame(requests.get(W_URL).json()["hourly"])
wx["time"] = pd.to_datetime(wx["time"])

df = aq.merge(wx, on="time", how="inner").sort_values("time").reset_index(drop=True)
df.to_csv("delhi_pm25_weather_hourly.csv", index=False)
print(len(df), "rows")

8664 rows


## 2. Random forest (our final model)
A random forest is many decision trees voting together, and it can catch patterns a straight-line model misses.

In [3]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1)
rf.fit(train[features], train["target"])
rf_pred = rf.predict(test[features])
print(f"Random forest error: {mean_absolute_error(test['target'], rf_pred):.1f} µg/m³")

Random forest error: 16.3 µg/m³


## 3. Where the model fails
These are the five hours where the forecast was furthest off. The model misses sudden changes, such as the air clearing quickly on 17 Oct and evening spikes on 3-4 Nov, because it only sees current conditions.

In [4]:
test = test.assign(pred=rf_pred, error=np.abs(test["target"] - rf_pred))
print(test.sort_values("error", ascending=False)[["time","pm2_5","target","pred","error"]].head(5))

                    time  pm2_5  target        pred      error
8089 2026-10-17 04:00:00  175.1    64.0  146.662763  82.662763
8085 2026-10-17 00:00:00  154.6   198.8  130.983233  67.816767
8532 2026-11-04 15:00:00   97.8   184.4  117.237798  67.162202
8508 2026-11-03 15:00:00   84.8   152.7   85.750957  66.949043
8533 2026-11-04 16:00:00  105.7   184.9  121.121652  63.778348


## 4. Purifier rule
The purifier switches ON when the forecast is above a trigger level. "Bad air" means PM2.5 above 60 µg/m³ six hours later. We assume the purifier uses a constant 60 W. The forecast is for outdoor air, while a purifier works indoors.

In [5]:
WATTS = 60        # your purifier's power
LIMIT = 60        # "bad air" line in µg/m³

# Each row: the model's guess for 6 hours ahead (rf_pred) vs what really happened (target)
on_model = rf_pred > LIMIT                 # purifier ON if the forest predicts bad air
on_lazy  = test["pm2_5"].values > LIMIT    # lazy rule: ON if the air is bad right now
bad      = test["target"].values > LIMIT   # was the air really bad?

def score(name, on):
    hours = len(on)
    caught = (on & bad).sum()
    missed = (~on & bad).sum()
    wasted = (on & ~bad).sum()
    kwh = on.sum() * WATTS / 1000
    kwh_always = hours * WATTS / 1000
    print(f"{name}")
    print(f"  Bad-air hours covered: {caught} of {bad.sum()}  (missed {missed})")
    print(f"  Hours ON when air was fine (wasted): {wasted}")
    print(f"  Energy used: {kwh:.1f} kWh vs {kwh_always:.1f} kWh if always on "
          f"(saved {kwh_always - kwh:.1f} kWh, {100*(1 - kwh/kwh_always):.0f}%)\n")

print(f"Test period: {len(test)} hours\n")
score("Model rule (forest forecast)", on_model)
score("Lazy rule (air right now)", on_lazy)

Test period: 1440 hours

Model rule (forest forecast)
  Bad-air hours covered: 756 of 867  (missed 111)
  Hours ON when air was fine (wasted): 155
  Energy used: 54.7 kWh vs 86.4 kWh if always on (saved 31.7 kWh, 37%)

Lazy rule (air right now)
  Bad-air hours covered: 669 of 867  (missed 198)
  Hours ON when air was fine (wasted): 198
  Energy used: 52.0 kWh vs 86.4 kWh if always on (saved 34.4 kWh, 40%)



## 5. Choosing the trigger level
A lower trigger misses fewer bad-air hours but saves less electricity. We chose **50 µg/m³** as a balance: it missed 38 of 867 bad hours and saved 22% of the energy. We chose it using the test period, so these numbers may be slightly optimistic.

In [6]:
for trigger in [30, 40, 50, 60, 70]:
    on = rf_pred > trigger
    caught = (on & bad).sum()
    missed = bad.sum() - caught
    wasted = (on & ~bad).sum()
    saved = 100 * (1 - on.sum() / len(on))
    print(f"Switch ON when forecast > {trigger}: missed {missed} bad hours, wasted {wasted}, energy saved {saved:.0f}%")

Switch ON when forecast > 30: missed 0 bad hours, wasted 519, energy saved 4%
Switch ON when forecast > 40: missed 1 bad hours, wasted 447, energy saved 9%
Switch ON when forecast > 50: missed 38 bad hours, wasted 298, energy saved 22%
Switch ON when forecast > 60: missed 111 bad hours, wasted 155, energy saved 37%
Switch ON when forecast > 70: missed 252 bad hours, wasted 79, energy saved 52%
